# 01 · Send, inspect, and receive your first message

**Level:** beginner · **Time:** 25–35 minutes · **Prerequisite:** finish notebook 00 and start RabbitMQ.

By the end, you can explain a producer, broker, exchange, queue, consumer, channel, and acknowledgement;
send Python data as JSON; and prove that a queue holds a message while no consumer is running.

Run cells in order. All examples finish by themselves. A fresh run uses a unique queue, so two presenters
can run this lesson without taking each other's messages. Run the cleanup cell when finished.

## 1. Follow the message

```text
Python producer → exchange → queue → Python consumer → acknowledgement
```

The broker is the RabbitMQ server. A connection is a network conversation with it; channels are logical
conversations inside that connection. An exchange chooses queues using bindings. A queue stores deliveries
until consumers handle them. An acknowledgement tells the broker a delivery can be removed.

Here we use the built-in **default exchange**, whose name is the empty string. RabbitMQ automatically
binds every queue to this exchange with its queue name as the routing key. We will create our own
exchanges in notebook 03. The helper only reads connection settings; the messaging operations stay visible.


In [ ]:
from pathlib import Path
import sys
import json
import time
import pika

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course" / "rabbit.py").is_file()), None)
assert ROOT is not None, "Open Jupyter from the RabbitMQ course folder."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message


## 2. Declare a queue

Predict: does creating an empty queue create a message? `message_count` should be zero.
This first queue is deliberately non-durable; later lessons address restart survival. Closing a connection
does not delete this queue because it is neither exclusive nor auto-delete.


In [ ]:
queue_name = unique_name("hello")
with connect() as connection:
    channel = connection.channel()
    result = channel.queue_declare(queue=queue_name, durable=False)
    print("Queue:", result.method.queue)
    print("Ready messages:", result.method.message_count)
    assert result.method.message_count == 0


## 3. Publish a JSON document, with no consumer running

Python dictionaries are not an AMQP wire format. Serialize to JSON and encode text as UTF-8 bytes.
`content_type` describes the payload; it does not perform serialization for you. The producer connection
will close at the end of this cell. Predict whether the message will still be available in the next cell.


In [ ]:
order = {"order_id": 1001, "customer": "Asha", "items": ["book", "pen"]}
with connect() as connection:
    channel = connection.channel()
    channel.basic_publish(
        exchange="", routing_key=queue_name,
        body=json.dumps(order).encode("utf-8"),
        properties=pika.BasicProperties(content_type="application/json", content_encoding="utf-8"),
    )
    # This synchronous inspection also lets us observe the queued message.
    ready = channel.queue_declare(queue=queue_name, passive=True).method.message_count
    print("Ready messages after publish:", ready)
    assert ready == 1


Pause to demonstrate: open the RabbitMQ management UI at `http://localhost:15672`, choose the
`learning` virtual host, open **Queues and Streams**, and find the printed queue name. Observe one
ready message and zero consumers. Do not use the UI's Get messages action yet: it can consume messages.

## 4. Read, process, then acknowledge

`basic_get` fetches at most one message. We use it here so a cell always finishes. A push consumer using
`basic_consume` is the usual approach for a running service. The wait helper retries `basic_get` until
a deadline and returns the same method, properties, and bytes as Pika.

Predict: the bytes become the original dictionary; after acknowledging, the queue will be empty.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    method, properties, body = wait_for_message(channel, queue_name, timeout=5)
    received = json.loads(body.decode("utf-8"))
    print("Raw bytes:", body)
    print("Decoded order:", received)
    print("Content type:", properties.content_type)
    assert received == order
    # Ack on the same channel that delivered this message, after successful work.
    channel.basic_ack(delivery_tag=method.delivery_tag)
    method, properties, body = channel.basic_get(queue=queue_name, auto_ack=False)
    assert method is None
    print("The queue is now empty.")


## 5. Try it yourself

Send three orders with IDs 1002–1004. Receive each one, collect the IDs, and acknowledge it only after
decoding. Before running your solution, predict the number of messages remaining after reading two.
The solution below expects an empty queue and creates its own three messages.


In [ ]:
# Suggested solution: change these payloads, then rerun this whole cell.
with connect() as connection:
    channel = connection.channel()
    for order_id in range(1002, 1005):
        channel.basic_publish(exchange="", routing_key=queue_name,
                              body=json.dumps({"order_id": order_id}).encode())
    seen = []
    for _ in range(2):
        method, _, body = wait_for_message(channel, queue_name)
        seen.append(json.loads(body)["order_id"])
        channel.basic_ack(method.delivery_tag)
    remaining = channel.queue_declare(queue=queue_name, passive=True).method.message_count
    assert remaining == 1
    method, _, body = wait_for_message(channel, queue_name)
    seen.append(json.loads(body)["order_id"])
    channel.basic_ack(method.delivery_tag)
    assert seen == [1002, 1003, 1004]
    print("Processed:", seen, "| Ready after the first two:", remaining)


## 6. Clean up and explain it back

Delete only this lesson's unique queue. Then explain why a message survived closing the producer
connection and why that does **not** prove it would survive a broker restart.


In [ ]:
with connect() as connection:
    connection.channel().queue_delete(queue=queue_name)
print("Removed", queue_name)


**Troubleshooting:** connection refused means the broker is not listening; check notebook 00.
`ACCESS_REFUSED` usually means credentials or virtual-host permissions do not match `.env`.
A receive timeout often means a previous run or management UI already consumed the message: restart
the kernel and run this notebook from the top. A queue-declaration conflict means an existing queue
was declared with different settings; unique names prevent that in normal use.

**Checkpoint:** you can distinguish storage (queue), routing (exchange), payload (bytes), and processing
confirmation (consumer acknowledgement). Next: multiple workers and recovery from a failed consumer.

Official reference: [RabbitMQ Python hello-world tutorial](https://www.rabbitmq.com/tutorials/tutorial-one-python).
